### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS
import os
from nequip.model import model_from_config


default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cuda:0',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

os.environ['NEQUIP_NUM_TASKS'] = '4'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./config/example_ETN_opt_MEA.yaml', defaults=default_config)
    

dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

dataset[0]

AtomicData(atom_types=[54, 1], cell=[3, 3], edge_cell_shift=[1354, 3], edge_index=[2, 1354], forces=[54, 3], pbc=[3], pos=[54, 3], stress=[3, 3], total_energy=[1])

In [3]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
#Nc = 10 # number of chennels for F features from ETN paper
#N_rank_spec = 4 # hidden rank of reduction for type radial tensor
#config['Nc'] = Nc
#config['N_rank_spec'] = N_rank_spec

# ETN parameters
#config['d'] = 4 # dimention of the tensor train
#config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/MEA_Allegro_ALS/example/log
  ...open log file results/MEA_Allegro_ALS/example/log
  ...generate file name results/MEA_Allegro_ALS/example/metrics_epoch.csv
  ...open log file results/MEA_Allegro_ALS/example/metrics_epoch.csv
  ...generate file name results/MEA_Allegro_ALS/example/metrics_initialization.csv
  ...open log file results/MEA_Allegro_ALS/example/metrics_initialization.csv
  ...generate file name results/MEA_Allegro_ALS/example/metrics_batch_train.csv
  ...open log file results/MEA_Allegro_ALS/example/metrics_batch_train.csv
  ...generate file name results/MEA_Allegro_ALS/example/metrics_batch_val.csv
  ...open log file results/MEA_Allegro_ALS/example/metrics_batch_val.csv
  ...generate file name results/MEA_Allegro_ALS/example/best_model.pth
  ...generate file name results/MEA_Allegro_ALS/example/last_model.pth
  ...generate file name results/MEA_Allegro_ALS/example/trainer.pth
  ...generate file name results/ME

  New scales: [Nb: 1.000000, Mo: 1.000000, Ta: 1.000000, W: 1.000000] shifts: [Nb: -13.299362, Mo: -13.299362, Ta: -13.299362, W: -13.299362]


In [4]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[0])

In [5]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math

# forward pass
data_new = final_model(data0)



### Main module

I'll skip the implementation of feature vector F because it heavily relies on nequip code for neighbor lists operations.
If you want you can look into the following files or ask me to add it to the notebook.

Otherwise look into this file

allegro/modules/ETN.py 

and corresponding layers


In [7]:
from typing import Optional, List
import math
import functools

import torch
from torch import nn
from torch_runstats.scatter import scatter

from e3nn import o3
from e3nn.util.jit import compile_mode

from nequip.data import AtomicDataDict
from nequip.nn import GraphModuleMixin
from nequip.utils.tp_utils import tp_path_exists

from allegro.nn._fc import ScalarMLPFunction
from allegro import _keys

from allegro.nn._strided import Contracter_ETN_ALS
from allegro.nn.cutoffs import cosine_cutoff, polynomial_cutoff
from e3nn.o3 import wigner_3j
from torch.nn import Parameter, ParameterList, ModuleList
from allegro.nn._edge_features_F import EdgeFeatures_FFunction

# Triangular ineguality for path existance
def tri_ineq(l1, l2, l3):
    return max([l1, l2, l3]) <= min([l1 + l2, l2 + l3, l1 + l3])


@compile_mode("script")
class ETN_ALS_A_B_Module_opt(nn.Module, GraphModuleMixin):
    def __init__(self,
                 d: int,
                 N_rank_ett: List[int],
                 Nc: int = 10,
                 num_types: int  = 3,
                 num_basis: int = 8,
                 N_rank_spec: int = 4,
                 avg_num_neighbors: Optional[float] = None,
                 normalize_edge_features_f: bool = True,
                 irreps_in=None,
                 out_field: str = AtomicDataDict.PER_ATOM_ENERGY_KEY
                ):
        
        super().__init__()
        self.out_field = out_field
        
        
        self.d = d
        self.Nc = Nc
        self.register_buffer("N_rank_ett", torch.as_tensor(N_rank_ett, dtype=torch.long))
        
        # set up irreps
        self._init_irreps(
            irreps_in=irreps_in,
            required_irreps_in=[
            ],
            irreps_out={_keys.NODE_FEATURES_ETN: o3.Irreps(
                    [(self.Nc, ir) for _, ir in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY] ]),
                        out_field: o3.Irreps([(1, (0, 1))])}
        )
        
        
        # Parameters of the network
        
        # tensors for atomic features encoding
        lmax = irreps_in[AtomicDataDict.EDGE_ATTRS_KEY].lmax # maximum spherical harmonic
        self.lmax = lmax
        
        # Second order cores(first and last)
        core2_1 = Parameter(torch.empty(lmax+1, 1, self.Nc, N_rank_ett[0]).normal_())
        core2_d = Parameter(torch.empty(lmax+1, N_rank_ett[-1], self.Nc, 1).normal_())


        instructions_1 = [(0, l, l) for l in range(lmax + 1)]
        instructions_d = [(l, l, 0) for l in range(lmax + 1)]
        
        
        # Third order cores
        # Assume irreps does not change 
        base_in1 = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        base_in2 = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        base_out = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        

        # Building instructions
        instructions: List[Tuple[int, int, int]] = []
        tmp_i_out: int = 0
        for i_out, (_, ir_out) in enumerate(base_out):
            for i_1, (_, ir_in1) in enumerate(base_in1):
                for i_2, (_, ir_in2) in enumerate(base_in2):
                    if ir_out in ir_in1 * ir_in2:
                        instructions.append((i_1, i_2, i_out))
        
                        tmp_i_out += 1

                        
        self.instructions = instructions
        
        self.register_buffer("instructions_list_0", torch.as_tensor(instructions_1, dtype = torch.long))
        for i in range(1, d - 1):
            self.register_buffer(f"instructions_list_{i}", torch.as_tensor(instructions, dtype = torch.long))
        self.register_buffer(f"instructions_list_{d - 1}", torch.as_tensor(instructions_d, dtype = torch.long))
        
        # building large w3j
        w3j_values = []
        w3j_index = []
        for i_in1, i_in2, i_out in instructions:
            mul_ir_in1 = base_in1[i_in1]
            mul_ir_in2 = base_in2[i_in2]
            mul_ir_out = base_out[i_out]
    
            assert mul_ir_in1.ir.p * mul_ir_in2.ir.p == mul_ir_out.ir.p
            assert (
                tri_ineq(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            )
    
            if mul_ir_in1.dim == 0 or mul_ir_in2.dim == 0 or mul_ir_out.dim == 0:
                raise ValueError
    
            this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            this_w3j_index = this_w3j.nonzero()
            w3j_values.append(
                this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
            )
    
            
            this_w3j_index[:, 0] += base_in1[: i_in1].dim
            this_w3j_index[:, 1] += base_in2[: i_in2].dim
            this_w3j_index[:, 2] += base_out[: i_out].dim
            # Now need to flatten the index to be for [pk][ij]
            w3j_index.append(
                torch.cat(
                    (   this_w3j_index[:, 2].unsqueeze(-1),
                        this_w3j_index[:, 0].unsqueeze(-1) * base_in2.dim
                        + this_w3j_index[:, 1].unsqueeze(-1),
                    ),
                    dim=1,
                )
            )
    
        num_paths: int = len(instructions)
    
        w3j = torch.sparse_coo_tensor(
            indices=torch.cat(w3j_index, dim=0).t(),
            values=torch.cat(w3j_values, dim=0),
            size=(
                num_paths * base_out.dim,
                base_in1.dim * base_in2.dim,
            ),
        ).coalesce()
        
        # in dense, must shape it for einsum:
        kij_shape = (
            base_out.dim,
            base_in1.dim,
            base_in2.dim,
        )
        
        # save to buffer in sparce mode + shape
        self.register_buffer("w3j", w3j)
        self.w3j_shape = (num_paths, ) + kij_shape
        
        # third order free parameters
        self.cores = ParameterList([core2_1] + [Parameter(torch.empty(num_paths, N_rank_ett[r], self.Nc, N_rank_ett[r+1]).normal_()) for r in range(d - 2)] + [core2_d]) 
        
        #self.reset_parameters()


        # Register layers F
        self.edge_F = ModuleList([EdgeFeatures_FFunction(
            lmax=self.lmax,
            num_types=num_types,
            Nc=self.Nc,
            num_basis=num_basis,
            N_rank_spec=N_rank_spec,
            irreps_edge_sh=base_in2,
        )  for r in range(self.d)])

        # To convert to node features
        if normalize_edge_features_f and avg_num_neighbors is not None:
            self._factor = 1.0 / math.sqrt(avg_num_neighbors)
        
        # Register layers tensor
        self.tps = [Contracter_ETN_ALS(base_in1, 
                                   N_rank_ett[r], 
                                   base_in2, 
                                   self.Nc, 
                                   base_out, 
                                   N_rank_ett[r+1], 
                                   num_paths) for r in range(self.d - 2)]

        
        
    def forward(self, data: AtomicDataDict.Type) -> AtomicDataDict.Type:

        edge_center = data[AtomicDataDict.EDGE_INDEX_KEY][0]
        edge_neighbor = data[AtomicDataDict.EDGE_INDEX_KEY][1]
        species = data[AtomicDataDict.ATOM_TYPE_KEY].squeeze(-1)
        
        # Input features
        edge_features_f = self.edge_F[-1](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                             data[_keys.EDGE_TYPE_KEY],
                             data[AtomicDataDict.EDGE_ATTRS_KEY])
        
        F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
        factor: Optional[float] = self._factor  # torchscript hack for typing
        if factor is not None:
            F = F * factor
        
        # Defining tensors for TorchScript
        u_out = torch.zeros((F.shape[0], F.shape[1], self.N_rank_ett[-1]), dtype=F.dtype,
            device=F.device) # temporary verctor output of etn
            
        
        data[_keys.NODE_FEATURES_ETN] = torch.zeros_like(F, dtype=F.dtype,
            device=F.device) # final feature output
        
        slices = self.irreps_in[AtomicDataDict.EDGE_ATTRS_KEY].slices() # slices over irreps

        # getting w3j in dense mode
        w3j_dense = (
            self.w3j.to_dense()
            .reshape(self.w3j_shape)
            .contiguous()
        )   
        
        # First transform using second order tensors
        for i, slice in enumerate(slices):
            u_out[:, slice, :] = torch.einsum('ij,Nmj->Nmi', self.cores[-1][i].squeeze(-1), F[:, slice, :])

        # Series third order tensors
        for i in range(self.d - 2, 0, -1):
            # Computing F
            edge_features_f = self.edge_F[i](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                     data[_keys.EDGE_TYPE_KEY],
                     data[AtomicDataDict.EDGE_ATTRS_KEY])
        
            F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
            if factor is not None:
                F = F * factor

            
            # big contruction
            u_out = self.tps[i-1](u_out, F, w3j_dense, self.cores[i])

        # Last transform using second order tensor
        for i, slice in enumerate(slices):
            data[_keys.NODE_FEATURES_ETN][:, slice, :] = torch.einsum('ij,Nmj->Nmi', self.cores[0][i].squeeze(0), u_out[:, slice, :])
        
        
        # Reduction to scalar
        # Computing F
        edge_features_f = self.edge_F[0](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                 data[_keys.EDGE_TYPE_KEY],
                 data[AtomicDataDict.EDGE_ATTRS_KEY])
    
        F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
        if factor is not None:
            F = F * factor

        
        data[self.out_field] = (( data[_keys.NODE_FEATURES_ETN] * F ).sum(dim = (-2, -1) )).unsqueeze(-1)
        

        return data

### Usage Example

In [103]:
torch.manual_seed(180)

ETN = ETN_ALS_A_B_Module_opt(d = config['d'],
                             N_rank_ett = config['N_rank_ett'],
                             Nc = config['Nc'],
                             num_types = config['num_types'],
                             num_basis = 8,
                             N_rank_spec = config['N_rank_spec'],
                             avg_num_neighbors=config['avg_num_neighbors'],
                             normalize_edge_features_f = True,
                     irreps_in = final_model.irreps_out,
                     out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY)

In [104]:
instr = [ETN.instructions_list_0, ETN.instructions_list_1, ETN.instructions_list_2, ETN.instructions_list_3]
R = [1] + ETN.N_rank_ett.tolist() + [1]

cores = ETN.cores

In [105]:
data_new_new_ref = ETN(data_new)

In [107]:
ref = [data_new_new_ref['atomic_energy'][0],
       data_new_new_ref['atomic_energy'][1],
       data_new_new_ref['atomic_energy'][2]]

print(ref[0], '\n',
      ref[1], '\n',
      ref[2])

tensor([-1.7746e-06], grad_fn=<SelectBackward0>) 
 tensor([-5.3040e-07], grad_fn=<SelectBackward0>) 
 tensor([-1.9824e-06], grad_fn=<SelectBackward0>)


In [108]:
from ortho import lr_orthogonal

cores_new, R = lr_orthogonal(cores, R, instr)

torch.Size([3, 1, 10, 4])
torch.Size([3, 1, 10, 4])
torch.Size([3, 1, 10, 4])
torch.Size([11, 4, 10, 4])
torch.Size([11, 4, 10, 4])
torch.Size([11, 4, 10, 4])
torch.Size([11, 4, 10, 4])
torch.Size([11, 4, 10, 4])
torch.Size([11, 4, 10, 4])


In [109]:
ETN.cores[0] = cores_new[0]
ETN.cores[1] = cores_new[1]
ETN.cores[2] = cores_new[2]
ETN.cores[3] = cores_new[3]

In [110]:
data_new_new_orth = ETN(data_new)

In [111]:
orth = [data_new_new_orth['atomic_energy'][0], 
        data_new_new_orth['atomic_energy'][1],
        data_new_new_orth['atomic_energy'][2]]

print(orth[0], '\n',
      orth[1], '\n',
      orth[2])

tensor([-2.7847e-07], grad_fn=<SelectBackward0>) 
 tensor([-2.1914e-07], grad_fn=<SelectBackward0>) 
 tensor([-3.2073e-07], grad_fn=<SelectBackward0>)


In [112]:
ref

[tensor([-1.7746e-06], grad_fn=<SelectBackward0>),
 tensor([-5.3040e-07], grad_fn=<SelectBackward0>),
 tensor([-1.9824e-06], grad_fn=<SelectBackward0>)]

In [115]:
print((ref[0]/orth[0]), '\n',
      ref[1]/orth[1], '\n',
      ref[2]/orth[2],
      )

tensor([6.3727], grad_fn=<DivBackward0>) 
 tensor([2.4204], grad_fn=<DivBackward0>) 
 tensor([6.1810], grad_fn=<DivBackward0>)


In [124]:
(cores_new[0].flatten(0, 2).T @ cores_new[0].flatten(0, 2))

tensor([[ 3.0000e+00,  2.9802e-08,  1.4901e-08, -8.9407e-08],
        [ 2.9802e-08,  3.0000e+00, -1.1921e-07, -1.1921e-07],
        [ 1.4901e-08, -1.1921e-07,  3.0000e+00, -8.9407e-08],
        [-8.9407e-08, -1.1921e-07, -8.9407e-08,  3.0000e+00]],
       grad_fn=<MmBackward0>)

In [125]:
(cores_new[1].flatten(0, 2).T @ cores_new[1].flatten(0, 2))

tensor([[ 3.0000e+00,  3.3528e-08, -3.7253e-09,  1.4901e-08],
        [ 3.3528e-08,  3.0000e+00,  0.0000e+00,  7.4506e-09],
        [-3.7253e-09,  0.0000e+00,  3.0000e+00,  1.8626e-08],
        [ 1.4901e-08,  7.4506e-09,  1.8626e-08,  3.0000e+00]],
       grad_fn=<MmBackward0>)

In [126]:
(cores_new[2].flatten(0, 2).T @ cores_new[2].flatten(0, 2))

tensor([[ 3.0000e+00, -1.8626e-08, -9.3132e-10, -7.4506e-09],
        [-1.8626e-08,  3.0000e+00,  2.5611e-08, -2.4214e-08],
        [-9.3132e-10,  2.5611e-08,  3.0000e+00, -5.5879e-09],
        [-7.4506e-09, -2.4214e-08, -5.5879e-09,  3.0000e+00]],
       grad_fn=<MmBackward0>)

In [127]:
(cores_new[3].flatten(0, 2).T @ cores_new[3].flatten(0, 2))

tensor([[17904.7090]], grad_fn=<MmBackward0>)